In [1]:
from sklearn.impute import SimpleImputer
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('..'))
from config.settings import DATA_PATH_RAW, DATA_PATH_CLEAN
import warnings
warnings.filterwarnings('ignore')

In [2]:
data_raw = pd.read_csv(DATA_PATH_RAW)
print(f"Dimensión de los datos {data_raw.shape}")
data_raw.head()

Dimensión de los datos (887379, 74)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,1077501,1296599,5000.0,5000.0,4975.0,36 months,10.65,162.87,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1077430,1314167,2500.0,2500.0,2500.0,60 months,15.27,59.83,C,C4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1077175,1313524,2400.0,2400.0,2400.0,36 months,15.96,84.33,C,C5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1076863,1277178,10000.0,10000.0,10000.0,36 months,13.49,339.31,C,C1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1075358,1311748,3000.0,3000.0,3000.0,60 months,12.69,67.79,B,B5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
data_raw[['loan_status']].isna().sum()

loan_status    0
dtype: int64

In [4]:
faltantes_80pct = (data_raw
        .isnull()
        .sum()
        .to_frame("Conteo_Faltantes")
        .assign(pct_faltantes = lambda x: round(100*(x.Conteo_Faltantes/len(data_raw)),4))
        .query("pct_faltantes > 80")
)

In [5]:
data_clean = data_raw.drop(columns=list(faltantes_80pct.index.values)+['id', 'member_id', 'title','emp_title','next_pymnt_d', 'policy_code'])
print(f"Se removieron {data_raw.shape[1]-data_clean.shape[1]} columnas")

Se removieron 25 columnas


In [6]:
faltantes_0_10 =(data_clean
        .isnull()
        .sum()
        .to_frame("Conteo_Faltantes")
        .assign(pct_faltantes = lambda x: round(100*(x.Conteo_Faltantes/len(data_clean)),4))
        .query("pct_faltantes > 0 & pct_faltantes<=10")
        .sort_values(by = "pct_faltantes")
)
faltantes_10plus =(data_clean
        .isnull()
        .sum()
        .to_frame("Conteo_Faltantes")
        .assign(pct_faltantes = lambda x: round(100*(x.Conteo_Faltantes/len(data_clean)),4))
        .query("pct_faltantes > 10")
        .sort_values(by = "pct_faltantes")
)

In [7]:
data_clean[faltantes_0_10.index.values]

,annual_inc,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,total_acc,acc_now_delinq,last_credit_pull_d,collections_12_mths_ex_med,revol_util,last_pymnt_d,emp_length,tot_coll_amt,tot_cur_bal,total_rev_hi_lim
0,24000.0,0.0,Jan-1985,1.0,3.0,0.0,9.0,0.0,Jan-2016,0.0,83.7,Jan-2015,10+ years,NaN,NaN,NaN
1,30000.0,0.0,Apr-1999,5.0,3.0,0.0,4.0,0.0,Sep-2013,0.0,9.4,Apr-2013,< 1 year,NaN,NaN,NaN
2,12252.0,0.0,Nov-2001,2.0,2.0,0.0,10.0,0.0,Jan-2016,0.0,98.5,Jun-2014,10+ years,NaN,NaN,NaN
3,49200.0,0.0,Feb-1996,1.0,10.0,0.0,37.0,0.0,Jan-2015,0.0,21.0,Jan-2015,10+ years,NaN,NaN,NaN
4,80000.0,0.0,Jan-1996,0.0,15.0,0.0,38.0,0.0,Jan-2016,0.0,53.9,Jan-2016,1 year,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887374,31000.0,0.0,Sep-2004,0.0,9.0,1.0,15.0,0.0,Jan-2016,0.0,82.1,Jan-2016,8 years,0.0,25274.0,17100.0
887375,79000.0,0.0,Mar-1974,1.0,5.0,0.0,23.0,0.0,Jan-2016,0.0,84.5,Jan-2016,10+ years,0.0,140285.0,10200.0
887376,35000.0,0.0,Sep-2003,0.0,9.0,1.0,22.0,0.0,Jan-2016,0.0,61.3,Jan-2016,5 years,0.0,34178.0,18000.0
887377,64400.0,1.0,Oct-2003,2.0,17.0,0.0,20.0,0.0,Jan-2016,1.0,30.6,Jan-2016,1 year,0.0,58418.0,27000.0


In [8]:
num_cols = data_clean[faltantes_0_10.index.values].select_dtypes(include=['number']).columns
obj_cols = data_clean[faltantes_0_10.index.values].select_dtypes(include=['object']).columns

imputer_mode = SimpleImputer(strategy='most_frequent')
imputer_mean = SimpleImputer(strategy='mean')

data_clean[obj_cols] = imputer_mode.fit_transform(data_clean[obj_cols])
data_clean[num_cols] = imputer_mean.fit_transform(data_clean[num_cols])

In [9]:
imputer_max1 = SimpleImputer(strategy='constant', fill_value=data_clean['mths_since_last_delinq'].max())
imputer_max2 = SimpleImputer(strategy='constant', fill_value=data_clean['mths_since_last_major_derog'].max())

data_clean['mths_since_last_delinq'] = imputer_max1.fit_transform(data_clean[['mths_since_last_delinq']])
data_clean['mths_since_last_major_derog'] = imputer_max2.fit_transform(data_clean[['mths_since_last_major_derog']])

In [10]:
data_clean[faltantes_10plus.index.values]

,mths_since_last_delinq,mths_since_last_major_derog
0,188.0,188.0
1,188.0,188.0
2,188.0,188.0
3,35.0,188.0
4,38.0,188.0
...,...,...
887374,188.0,188.0
887375,26.0,29.0
887376,188.0,188.0
887377,22.0,22.0


In [11]:
print(f"Cantidad de celdas nulas: {data_clean.isnull().sum().sum()}")

Cantidad de celdas nulas: 0


In [12]:
#data_clean.to_csv(DATA_PATH_CLEAN,sep=',', decimal='.', index=False)
data_clean.to_parquet(DATA_PATH_CLEAN)